In [3]:
"""
train_sentence.py – ISL Sentence Model Training Pipeline (Optimised v2)
========================================================================

Key improvements over v1:
  1. Landmark normalisation  — centres & scales landmarks per-frame so
     body position / distance from camera no longer confuse the model
  2. Sequence augmentation   — time-warp, Gaussian noise, frame dropout,
     coordinate jitter; synthetically multiplies 40 samples into hundreds
  3. SEQUENCE_LEN = 45       — captures full signing motion better
  4. GRU + Conv1D model      — 3× fewer params, far less overfitting
  5. No label smoothing      — was actively hurting with < 20 samples/class
  6. Longer patience (15)    — small datasets need more epochs to converge
  7. Cache invalidation flag — set FORCE_REEXTRACT=True after changing
     SEQUENCE_LEN or normalisation to rebuild .npy cache

Run:
    python train_sentence.py
"""

import os
import sys
import json
import random
import importlib
import numpy as np
import cv2
import tensorflow as tf
import tensorflow as tf
EarlyStopping = tf.keras.callbacks.EarlyStopping
ModelCheckpoint = tf.keras.callbacks.ModelCheckpoint
ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

sys.path.insert(0, "C:\\Users\\kkani\\Documents\\py files\\ISL")
from sentence.sentence_model import (
    build_sentence_model, print_sentence_model_summary,
    SEQUENCE_LEN, FEATURE_DIM,
    SENTENCE_MODEL_PATH, SENTENCE_LABEL_PATH
)

# ── CONFIGURE ─────────────────────────────────────────────────────────────────
DATASET_PATH = r"C:\\Users\\kkani\\Downloads\\isl_sentence" 
CACHE_DIR       = "landmark_cache_v2"   # new cache dir — avoids stale v1 data
FORCE_REEXTRACT = False   # set True if you change SEQUENCE_LEN or normalisation
# ─────────────────────────────────────────────────────────────────────────────

RANDOM_SEED   = 42
TEST_SPLIT    = 0.20
EPOCHS        = 80        # more epochs — small data needs longer to converge
BATCH_SIZE    = 16        # slightly bigger — more stable gradients with augmentation
LEARNING_RATE = 5e-4      # matches sentence_model.py default
VALID_EXTS    = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

# Augmentation config
AUG_MULTIPLIER    = 6     # generate 6 synthetic copies per real sequence
NOISE_STD         = 0.005 # Gaussian noise std (landmark coords are in [0,1])
JITTER_STD        = 0.01  # per-frame coordinate jitter
TIME_WARP_MAX     = 0.15  # max fraction of frames to shift in time warp
FRAME_DROP_PROB   = 0.10  # probability of dropping (zeroing) a random frame


# ══════════════════════════════════════════════════════════════════════════════
# MediaPipe
# ══════════════════════════════════════════════════════════════════════════════
try:
    _mp_holistic = importlib.import_module("mediapipe.python.solutions.holistic")
except Exception as e:
    print(f"[ERROR] MediaPipe not found: {e}")
    print("Run: pip install mediapipe==0.10.13")
    sys.exit(1)


# ══════════════════════════════════════════════════════════════════════════════
# Landmark helpers
# ══════════════════════════════════════════════════════════════════════════════

def extract_landmarks_from_results(results) -> np.ndarray:
    """Flatten Holistic results → (FEATURE_DIM,) = (258,)."""
    def hand_vec(lm):
        if lm:
            return np.array([[l.x, l.y, l.z]
                             for l in lm.landmark]).flatten()
        return np.zeros(63)

    def pose_vec(lm):
        if lm:
            return np.array([[l.x, l.y, l.z, l.visibility]
                             for l in lm.landmark]).flatten()
        return np.zeros(132)

    return np.concatenate([
        hand_vec(results.left_hand_landmarks),
        hand_vec(results.right_hand_landmarks),
        pose_vec(results.pose_landmarks)
    ])


def normalise_sequence(seq: np.ndarray) -> np.ndarray:
    """
    Normalise a landmark sequence so predictions are robust to:
      • signers sitting/standing at different distances from the camera
      • camera height differences
      • body position left/right in the frame

    Method: for each frame, subtract the mean of all non-zero landmark
    coordinates and divide by their standard deviation.
    Frames where no hand/pose detected (all zeros) are left as zeros.

    Args:
        seq: shape (SEQUENCE_LEN, FEATURE_DIM)

    Returns:
        Normalised array of same shape.
    """
    out = seq.copy()
    for i, frame in enumerate(out):
        # Only normalise frames that have real data
        nonzero = frame[frame != 0]
        if len(nonzero) > 10:
            mu  = nonzero.mean()
            std = nonzero.std()
            if std > 1e-6:
                # Normalise non-zero values; keep zero-padding as zeros
                mask      = frame != 0
                out[i][mask] = (frame[mask] - mu) / std
    return out


def video_to_sequence(video_path: str,
                      holistic,
                      seq_len: int = SEQUENCE_LEN) -> np.ndarray | None:
    """
    Read video → extract per-frame landmarks → uniform temporal sample → normalise.
    Returns (seq_len, FEATURE_DIM) or None on failure.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    raw_frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        res = holistic.process(rgb)
        raw_frames.append(extract_landmarks_from_results(res))
    cap.release()

    if len(raw_frames) == 0:
        return None

    frames = np.array(raw_frames, dtype=np.float32)   # (T, 258)
    n      = len(frames)

    # Uniform temporal sampling / padding to fixed length
    if n >= seq_len:
        idxs = np.linspace(0, n - 1, seq_len, dtype=int)
        seq  = frames[idxs]
    else:
        pad = np.zeros((seq_len - n, FEATURE_DIM), dtype=np.float32)
        seq = np.vstack([frames, pad])

    return normalise_sequence(seq)


# ══════════════════════════════════════════════════════════════════════════════
# Sequence augmentation
# ══════════════════════════════════════════════════════════════════════════════

def augment_sequence(seq: np.ndarray) -> np.ndarray:
    """
    Apply random augmentations to a landmark sequence.
    All ops are applied in landmark-coordinate space — label-preserving.

    Augmentations:
      1. Gaussian noise       — mimics sensor jitter / tracking imprecision
      2. Coordinate jitter    — slight random shift per frame
      3. Time warp            — stretch/compress random temporal segment
      4. Frame dropout        — randomly zero out a frame (mimics occlusion)
      5. Horizontal flip      — mirror the sign (valid since ISL has both-hand signs)
      6. Scale jitter         — random scale ±10% (distance from camera)
    """
    aug = seq.copy()

    # 1. Gaussian noise
    aug += np.random.normal(0, NOISE_STD, aug.shape).astype(np.float32)

    # 2. Per-frame coordinate jitter
    jitter = np.random.normal(0, JITTER_STD,
                              (SEQUENCE_LEN, 1)).astype(np.float32)
    aug   += jitter

    # 3. Time warp — shift content of a random window by ±1 frame
    warp_len = max(1, int(SEQUENCE_LEN * TIME_WARP_MAX))
    # Ensure we have room for shifting in both directions
    max_start = SEQUENCE_LEN - warp_len - 2
    if max_start > 0:
        start     = random.randint(0, max_start)
        direction = random.choice([-1, 1])
        segment   = aug[start: start + warp_len].copy()
        
        # Calculate target position and clip to valid range
        target_start = start + direction
        target_end = target_start + warp_len
        
        if 0 <= target_start and target_end <= SEQUENCE_LEN:
            aug[target_start: target_end] = segment

    # 4. Frame dropout — zero out random frames
    for i in range(SEQUENCE_LEN):
        if random.random() < FRAME_DROP_PROB:
            aug[i] = 0.0

    # 5. Horizontal flip — negate x-coordinates of hands + pose
    #    Hand x-coords are at positions 0, 3, 6, … within 63-element blocks
    #    Pose x-coords are every 4th value in the pose block
    if random.random() < 0.5:
        # Left hand x (indices 0,3,6,…,60 in first 63 values)
        aug[:, 0:63:3]   = -aug[:, 0:63:3]
        # Right hand x (indices 63,66,…,123)
        aug[:, 63:126:3] = -aug[:, 63:126:3]
        # Pose x (indices 126,130,134,… every 4th starting at 126)
        aug[:, 126::4]   = -aug[:, 126::4]
        # Swap left/right hand blocks so the label still matches
        left_block  = aug[:, :63].copy()
        right_block = aug[:, 63:126].copy()
        aug[:, :63]    = right_block
        aug[:, 63:126] = left_block

    # 6. Scale jitter ±10%
    scale  = np.random.uniform(0.90, 1.10)
    aug   *= scale

    return aug


def augment_dataset(X: np.ndarray, y: np.ndarray,
                    multiplier: int = AUG_MULTIPLIER):
    """
    Expand dataset by generating `multiplier` synthetic copies per sample.
    Returns combined original + synthetic arrays.
    """
    X_aug, y_aug = [X], [y]
    print(f"[Augment] Generating {multiplier}× synthetic copies "
          f"({len(X) * multiplier} new samples) …")

    for _ in range(multiplier):
        batch = np.stack([augment_sequence(s) for s in X], axis=0)
        X_aug.append(batch)
        y_aug.append(y)

    X_out = np.concatenate(X_aug, axis=0)
    y_out = np.concatenate(y_aug, axis=0)

    # Shuffle so augmented copies don't end up in contiguous blocks
    perm  = np.random.permutation(len(X_out))
    return X_out[perm], y_out[perm]


# ══════════════════════════════════════════════════════════════════════════════
# Dataset builder
# ══════════════════════════════════════════════════════════════════════════════

def build_dataset(dataset_path: str):
    classes = sorted([
        d for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ])

    if not classes:
        raise ValueError(f"No class folders found in: {dataset_path}")

    label_map = {i: name for i, name in enumerate(classes)}
    os.makedirs(CACHE_DIR, exist_ok=True)

    X_all, y_all = [], []

    holistic = _mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,          # model_complexity=1 for better accuracy
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    print(f"\n[Train] Extracting landmarks — cache dir: '{CACHE_DIR}/'")
    print(f"        (first run is slow; subsequent runs use cached .npy files)\n")

    for cls_idx, cls_name in enumerate(classes):
        cls_dir = os.path.join(dataset_path, cls_name)
        videos  = [
            f for f in os.listdir(cls_dir)
            if os.path.splitext(f)[1].lower() in VALID_EXTS
        ]
        cls_seqs = []

        for vid_name in videos:
            safe_name  = (cls_name + "_" + vid_name).replace(" ", "_") \
                                                     .replace("/", "_") \
                                                     .replace("\\", "_")
            cache_path = os.path.join(CACHE_DIR,
                                      os.path.splitext(safe_name)[0] + ".npy")

            if not FORCE_REEXTRACT and os.path.exists(cache_path):
                seq = np.load(cache_path)
            else:
                seq = video_to_sequence(
                    os.path.join(cls_dir, vid_name), holistic
                )
                if seq is None:
                    continue
                np.save(cache_path, seq)

            cls_seqs.append(seq)

        if not cls_seqs:
            print(f"  [{cls_name}]  — no valid videos, skipping class.")
            continue

        print(f"  [{cls_idx:02d}] {cls_name:<35}  {len(cls_seqs):>3} sequences")
        X_all.extend(cls_seqs)
        y_all.extend([cls_idx] * len(cls_seqs))

    holistic.close()

    X = np.array(X_all, dtype=np.float32)
    y = np.array(y_all, dtype=np.int32)
    print(f"\n[Train] Raw dataset : {len(X)} sequences  shape={X.shape}")
    return X, y, label_map, classes


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════

def main():
    if not os.path.isdir(DATASET_PATH):
        print(f"[ERROR] Dataset not found:\n  {DATASET_PATH}")
        print("Run mini_dataset_sentences.py first.")
        sys.exit(1)

    import multiprocessing
    n_cores = multiprocessing.cpu_count()
    tf.config.threading.set_intra_op_parallelism_threads(n_cores)
    tf.config.threading.set_inter_op_parallelism_threads(n_cores)
    print(f"[Train] CPU mode — {n_cores} cores.")

    np.random.seed(RANDOM_SEED)
    random.seed(RANDOM_SEED)
    tf.random.set_seed(RANDOM_SEED)

    # ── Extract landmarks ──────────────────────────────────────────────────────
    X, y, label_map, class_names = build_dataset(DATASET_PATH)[:4]
    num_classes = len(class_names)

    # Save label map
    with open(SENTENCE_LABEL_PATH, "w") as f:
        json.dump({str(k): v for k, v in label_map.items()}, f, indent=2)
    print(f"[Train] Sentence label map saved → {SENTENCE_LABEL_PATH}")

    # ── Train / val split BEFORE augmentation ─────────────────────────────────
    # Critical: augmentation must only apply to training set.
    # Augmenting before split would leak synthetic copies into validation.
    X_train_raw, X_val, y_train_raw, y_val = train_test_split(
        X, y,
        test_size=TEST_SPLIT,
        random_state=RANDOM_SEED,
        stratify=y if num_classes > 1 else None
    )
    print(f"[Train] Before augmentation — Train: {len(X_train_raw)}  "
          f"Val: {len(X_val)}")

    # ── Augment training set only ──────────────────────────────────────────────
    X_train, y_train = augment_dataset(X_train_raw, y_train_raw,
                                       multiplier=AUG_MULTIPLIER)
    print(f"[Train] After augmentation  — Train: {len(X_train)}  "
          f"Val: {len(X_val)}")

    # ── One-hot encode ─────────────────────────────────────────────────────────
    Y_train = tf.keras.utils.to_categorical(y_train, num_classes)
    Y_val   = tf.keras.utils.to_categorical(y_val,   num_classes)

    # ── Class weights ──────────────────────────────────────────────────────────
    weights_arr   = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=y_train
    )
    class_weights = {i: w for i, w in enumerate(weights_arr)}

    # ── Build model ────────────────────────────────────────────────────────────
    print(f"\n[Train] Building GRU model for {num_classes} sentence classes …")
    model = build_sentence_model(num_classes, learning_rate=LEARNING_RATE)
    print_sentence_model_summary(model)

    # ── Callbacks ──────────────────────────────────────────────────────────────
    callbacks = [
        ModelCheckpoint(
            filepath=SENTENCE_MODEL_PATH,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
            mode="max"
        ),
        EarlyStopping(
            monitor="val_accuracy",
            patience=15,             # more patience — small data is noisy
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=7,
            min_lr=1e-7,
            verbose=1
        ),
    ]

    # ── Train ──────────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"Training GRU  ({EPOCHS} epochs max, early stopping patience=15)")
    print(f"{'='*60}\n")

    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )

    # ── Per-class accuracy ─────────────────────────────────────────────────────
    print("\n[Train] Per-class accuracy on UNAUGMENTED validation set:")
    preds        = model.predict(X_val, verbose=0)
    pred_classes = np.argmax(preds, axis=1)

    print(f"\n  {'Sentence':<35} {'Correct':>8} {'Total':>8} {'Acc':>8}")
    print("  " + "-"*64)
    for i, name in enumerate(class_names):
        mask    = y_val == i
        total   = mask.sum()
        correct = (pred_classes[mask] == i).sum()
        acc     = correct / total * 100 if total > 0 else 0
        flag    = "  ← LOW" if acc < 50 and total > 0 else ""
        print(f"  {name:<35} {correct:>8} {total:>8} {acc:>7.1f}%{flag}")

    overall = (pred_classes == y_val).mean() * 100
    print(f"\n  Overall val accuracy : {overall:.2f}%")

    if overall < 40:
        print("\n  [HINT] Accuracy still low. Check:")
        print("         1. Do you have 50 videos per class in the mini dataset?")
        print("         2. Are the videos clearly shot with good lighting?")
        print("         3. Try increasing AUG_MULTIPLIER to 8 in train_sentence.py")
    elif overall < 65:
        print("\n  [HINT] Decent start — try increasing VIDEOS_PER_CLASS to 80+ "
              "in mini_dataset_sentences.py for better accuracy.")
    else:
        print("\n  [OK] Good accuracy! Run python predict.py")

    print(f"\n  Model saved → {SENTENCE_MODEL_PATH}")
    print("  Run  python predict.py  to start live detection.\n")


if __name__ == "__main__":
    main()

[Train] CPU mode — 12 cores.

[Train] Extracting landmarks — cache dir: 'landmark_cache_v2/'
        (first run is slow; subsequent runs use cached .npy files)

  [00] Are you Free Today                    51 sequences
  [01] Can you repeat that please            51 sequences
  [02] Congratulations                       50 sequences
  [03] Help Me Please                        51 sequences
  [04] I am fine                             48 sequences
  [05] I love you                            51 sequences
  [06] Please come,Welcome                   52 sequences
  [07] Talk slower please                    50 sequences
  [08] Thank You                             49 sequences
  [09] What Happened                         50 sequences
  [10] What are you doing                    50 sequences
  [11] What do you do                        51 sequences
  [12] how are you                           50 sequences
  [13] no                                    50 sequences
  [14] yes                 

Model: "ISL_Sentence_GRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ landmark_sequence (InputLayer)  │ (None, 45, 258)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 45, 64)         │        49,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 45, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 45, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 45, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 22, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 22, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 22, 128)        │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sentence_class (Dense)          │ (None, 15)             │           975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 179,599 (701.56 KB)

 Trainable params: 179,215 (700.06 KB)

 Non-trainable params: 384 (1.50 KB)


[SentenceModel] Trainable params: 179,215

Training GRU  (80 epochs max, early stopping patience=15)

Epoch 1/80
264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0820 - loss: 3.0293
Epoch 1: val_accuracy improved from None to 0.12583, saving model to isl_sentence_model.keras

Epoch 1: finished saving model to isl_sentence_model.keras
264/264 ━━━━━━━━━━━━━━━━━━━━ 21s 31ms/step - accuracy: 0.0950 - loss: 2.8823 - val_accuracy: 0.1258 - val_loss: 2.5235 - learning_rate: 5.0000e-04
Epoch 2/80
263/264 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.1304 - loss: 2.5369
Epoch 2: val_accuracy improved from 0.12583 to 0.22517, saving model to isl_sentence_model.keras

Epoch 2: finished saving model to isl_sentence_model.keras
264/264 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.1613 - loss: 2.4418 - val_accuracy: 0.2252 - val_loss: 2.0956 - learning_rate: 5.0000e-04
Epoch 3/80
262/264 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2431 - loss: 2.1767
Epoch 3: val_accuracy improved f